# Prepare Dataset

In [17]:
import json, string
import numpy as np

In [18]:
with open("data/alfred/train.json", "r") as file:
    train_dataset = json.load(file)

with open("data/alfred/valid_unseen.json", "r") as file:
    test_dataset_unseen = json.load(file)

with open("data/alfred/valid_seen.json", "r") as file:
    test_dataset_seen = json.load(file)

with open("data/stopwords.json", "r") as file:
    stopwords = set(json.load(file))

def text_to_words(text):
    return text.strip(string.punctuation + string.whitespace).lower().split()

# Method 1: Bag of Words (BoW)

For each label, consider all the task descriptions as its document content.
In BoW model, we simply count the occurences of each word, without considering the order.

In [19]:
from collections import defaultdict, Counter

# Construct the BoW for each label
BoW = defaultdict(Counter)
for item in train_dataset:
    BoW[item["label"]].update(text_to_words(item["task"]))

# Print the labels
for label in BoW.keys():
    print(label)

look_at_obj_in_light
pick_and_place_simple
pick_and_place_with_movable_recep
pick_clean_then_place_in_recep
pick_cool_then_place_in_recep
pick_heat_then_place_in_recep
pick_two_obj_and_place


## 1.1 BoW Probability

First we consider predicting by the probability of words in each labels. That is:

$$
P(\text{text}|\text{label}) 
= \prod_{\text{word}\in\text{text}} p(\text{label}|\text{word})
= \prod_{\text{word}\in\text{text}} \frac{\text{tf}_{\text{word},\text{label}}}{\text{tf}_{\text{word}}}
$$

Obviously for calculation simplicity we can simply compare:

$$
\begin{aligned}
&\log P(\text{text}|\text{label}) \\
=& \sum_{\text{word}\in\text{text}} \left( \log{\text{tf}_{\text{word},\text{label}}} -\log{\text{tf}_{\text{word}}} \right) \\
=& \sum_{\text{word}\in\text{text}} \log{\text{tf}_{\text{word},\text{label}}}\ 
\underbrace{-\sum_{\text{word}\in\text{text}} \log{\text{tf}_{\text{word}}}}_\text{common part}
\end{aligned}
$$

In [20]:
def bow_prob(text, remove_stopwords=False):
    words = text_to_words(text)
    label_log_probs = {}
    for label, counter in BoW.items():
        log_prob = 0.0
        for word in words:
            if remove_stopwords and word in stopwords:
                continue
            if counter[word] > 0:
                log_prob += np.log(counter[word])
        label_log_probs[label] = log_prob
    return max(label_log_probs, key=label_log_probs.get)

In [21]:
correct = 0
for item in test_dataset_unseen:
    label = item["label"]
    predict = bow_prob(item["task"], remove_stopwords=False)
    if label == predict:
        correct += 1
print(f"Unseen Dataset Accuracy: {correct / len(test_dataset_unseen)}")

Unseen Dataset Accuracy: 0.9013398294762485


In [22]:
correct = 0
for item in test_dataset_seen:
    label = item["label"]
    predict = bow_prob(item["task"], remove_stopwords=False)
    if label == predict:
        correct += 1
print(f"seen Dataset Accuracy: {correct / len(test_dataset_seen)}")

seen Dataset Accuracy: 0.8463414634146341


## Methods 2

**TF-IDF with Supervised Classifiers**

To represent the instruction text numerically, we adopt the Term Frequency–Inverse Document Frequency (TF-IDF) representation. 
TF-IDF reflects the importance of a word in a document relative to the entire corpus.

For a term $t$ in document $d$, the TF-IDF weight is defined as

\begin{equation}
\text{TF-IDF}(t,d) = TF(t,d) \times IDF(t)
\end{equation}

where the term frequency (TF) is defined as

\begin{equation}
TF(t,d) = \frac{f_{t,d}}{\sum_{t'} f_{t',d}}
\end{equation}

Here, $f_{t,d}$ denotes the number of occurrences of term $t$ in document $d$.

The inverse document frequency (IDF) is defined as

\begin{equation}
IDF(t) = \log \left(\frac{N}{df(t)}\right)
\end{equation}

where $N$ is the total number of documents in the corpus and $df(t)$ is the number of documents containing term $t$.

Each document is therefore represented as a TF-IDF feature vector

\begin{equation}
\mathbf{x}_d = [w_1, w_2, \dots, w_V]
\end{equation}

where $V$ denotes the vocabulary size and $w_i$ represents the TF-IDF weight of term $i$.

The resulting TF-IDF feature vectors are used as input to several supervised classifiers, including Logistic Regression, Random Forest, Support Vector Machines (SVM), and k-Nearest Neighbors (KNN), in order to predict the task label from the instruction text.

In [23]:
# Prepare train data
train_texts = [item["task"] for item in train_dataset]
train_labels = [item["label"] for item in train_dataset]

# validation (seen)
test_texts_seen = [item["task"] for item in test_dataset_seen]
test_labels_seen = [item["label"] for item in test_dataset_seen]

# validation (unseen)
test_texts_unseen = [item["task"] for item in test_dataset_unseen]
test_labels_unseen = [item["label"] for item in test_dataset_unseen]

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words=list(stopwords),
    max_features=5000,
    ngram_range=(1,2)
)

X_train = vectorizer.fit_transform(train_texts)

X_test_seen = vectorizer.transform(test_texts_seen)
X_test_unseen = vectorizer.transform(test_texts_unseen)

print("Train shape:", X_train.shape)
print("Seen shape:", X_test_seen.shape)
print("Unseen shape:", X_test_unseen.shape)

Train shape: (21025, 5000)
Seen shape: (820, 5000)
Unseen shape: (821, 5000)


In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score

log_model = LogisticRegression(max_iter=2000)

log_model.fit(X_train, train_labels)

pred_seen = log_model.predict(X_test_seen)
pred_unseen = log_model.predict(X_test_unseen)

acc_seen = accuracy_score(test_labels_seen, pred_seen)
acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

print("Logistic Regression")
print("Seen Accuracy:", acc_seen)
print("Unseen Accuracy:", acc_unseen)
print()

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, train_labels)

pred_seen = rf_model.predict(X_test_seen)
pred_unseen = rf_model.predict(X_test_unseen)

acc_seen = accuracy_score(test_labels_seen, pred_seen)
acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

print("Random Forest")
print("Seen Accuracy:", acc_seen)
print("Unseen Accuracy:", acc_unseen)
print()

svm_model = LinearSVC()

svm_model.fit(X_train, train_labels)

pred_seen = svm_model.predict(X_test_seen)
pred_unseen = svm_model.predict(X_test_unseen)

acc_seen = accuracy_score(test_labels_seen, pred_seen)
acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

print("SVM")
print("Seen Accuracy:", acc_seen)
print("Unseen Accuracy:", acc_unseen)
print()

knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train, train_labels)

pred_seen = knn_model.predict(X_test_seen)
pred_unseen = knn_model.predict(X_test_unseen)

acc_seen = accuracy_score(test_labels_seen, pred_seen)
acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

print("KNN")
print("Seen Accuracy:", acc_seen)
print("Unseen Accuracy:", acc_unseen)
print()

Logistic Regression
Seen Accuracy: 0.9195121951219513
Unseen Accuracy: 0.9464068209500609

Random Forest
Seen Accuracy: 0.9195121951219513
Unseen Accuracy: 0.9403166869671132

SVM
Seen Accuracy: 0.9280487804878049
Unseen Accuracy: 0.9451887941534713

KNN
Seen Accuracy: 0.8670731707317073
Unseen Accuracy: 0.8708891595615104



In [ ]:
results = {}

def evaluate(model, name):
    model.fit(X_train, train_labels)

    pred_seen = model.predict(X_test_seen)
    pred_unseen = model.predict(X_test_unseen)

    acc_seen = accuracy_score(test_labels_seen, pred_seen)
    acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

    results[name] = (acc_seen, acc_unseen)

evaluate(LogisticRegression(max_iter=2000), "Logistic")
evaluate(RandomForestClassifier(n_estimators=200), "RandomForest")
evaluate(LinearSVC(), "SVM")
evaluate(KNeighborsClassifier(n_neighbors=5), "KNN")

print("Model Comparison")
for k,v in results.items():
    print(k, "Seen:", round(v[0],4), "Unseen:", round(v[1],4))

Model Comparison
Logistic Seen: 0.9195 Unseen: 0.9464
RandomForest Seen: 0.9207 Unseen: 0.9367
SVM Seen: 0.928 Unseen: 0.9452
KNN Seen: 0.8671 Unseen: 0.8709


## Method 3

**Sentence Embedding with Pretrained Transformer**

To capture semantic information beyond word frequency features, we utilize a pretrained sentence embedding model to encode each task instruction into a dense vector representation.

Given an instruction text $d$, we obtain its embedding using a pretrained sentence encoder:

$$
\mathbf{x}_d = f(d)
$$

where $f(\cdot)$ denotes the pretrained sentence embedding model. In this work, we use the **all-MiniLM-L6-v2** model from Sentence-Transformers, which maps each instruction to a dense vector:

$$
\mathbf{x}_d \in \mathbb{R}^{384}
$$

The resulting embedding vector is then fed into a Multi-Layer Perceptron (MLP) classifier to predict the task label.

The hidden layer representation is computed as

$$
\mathbf{h} = \sigma(W_1 \mathbf{x}_d + b_1)
$$

where $W_1$ and $b_1$ are learnable parameters and $\sigma(\cdot)$ denotes the ReLU activation function.

The final output layer produces the probability distribution over task labels using a softmax function:

$$
\mathbf{y} = \text{softmax}(W_2 \mathbf{h} + b_2)
$$

where $\mathbf{y} \in \mathbb{R}^{C}$ and $C$ is the number of task classes.

In [27]:
!pip install sentence-transformers

  Obtaining dependency information for sentence-transformers from https://files.pythonhosted.org/packages/46/9f/dba4b3e18ebbe1eaa29d9f1764fbc7da0cd91937b83f2b7928d15c5d2d36/sentence_transformers-5.2.3-py3-none-any.whl.metadata
  Obtaining dependency information for transformers<6.0.0,>=4.41.0 from https://files.pythonhosted.org/packages/b8/88/ae8320064e32679a5429a2c9ebbc05c2bf32cefb6e076f9b07f6d685a9b4/transformers-5.3.0-py3-none-any.whl.metadata
  Obtaining dependency information for huggingface-hub>=0.20.0 from https://files.pythonhosted.org/packages/ec/74/2bc951622e2dbba1af9a460d93c51d15e458becd486e62c29cc0ccb08178/huggingface_hub-1.5.0-py3-none-any.whl.metadata
  Obtaining dependency information for tokenizers<=0.23.0,>=0.22.0 from https://files.pythonhosted.org/packages/65/71/0670843133a43d43070abeb1949abfdef12a86d490bea9cd9e18e37c5ff7/tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata
  Obtaining dependency information for hf-xet<2.0.0,>=1.2.0 from https://files.pythonhosted.org/

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
lerobot 0.4.3 requires huggingface-hub[cli,hf-transfer]<0.36.0,>=0.34.2, but you have huggingface-hub 1.5.0 which is incompatible.


In [28]:
from sentence_transformers import SentenceTransformer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import numpy as np

In [29]:
model = SentenceTransformer('all-MiniLM-L6-v2')

X_train_embed = model.encode(train_texts, show_progress_bar=True)

X_test_seen_embed = model.encode(test_texts_seen, show_progress_bar=True)

X_test_unseen_embed = model.encode(test_texts_unseen, show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\lijun\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lijun\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/658 [00:00<?, ?it/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

In [30]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

mlp_model = MLPClassifier(
    hidden_layer_sizes=(256,128),
    activation='relu',
    max_iter=50,
    random_state=42
)

mlp_model.fit(X_train_embed, train_labels)

c:\Users\lijun\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=50, random_state=42)

In [31]:
pred_seen = mlp_model.predict(X_test_seen_embed)

acc_seen = accuracy_score(test_labels_seen, pred_seen)

print("Sentence Embedding + MLP")
print("Seen Accuracy:", acc_seen)

Sentence Embedding + MLP
Seen Accuracy: 0.9378048780487804


In [33]:
pred_unseen = mlp_model.predict(X_test_unseen_embed)

acc_unseen = accuracy_score(test_labels_unseen, pred_unseen)

print("Unseen Accuracy:", acc_unseen)

Unseen Accuracy: 0.9269183922046285
